In [ ]:
!pip install open3d datasets plyfile torch huggingface_hub plotly

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 109.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 89.2 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
    Uninstalling ipywidgets-7.7.1:
      Successfully uninstalled ipywidgets-7.7.1


#Imports

In [ ]:
import open3d as o3d
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import plotly.graph_objects as go
from tqdm import tqdm

#Logging Hugging Face

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HUGGINGFACE")
from huggingface_hub import login

login(token=HF_TOKEN)

# Downloading the "BGLab/MaizeField3D" dataset

In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="BGLab/MaizeField3D",
    repo_type="dataset",
    local_dir="/content/maize_dataset"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

'/content/maize_dataset'

# Unzipping zipped file

In [ ]:
import zipfile
import os

zip_path = "/content/maize_dataset/datasets/FielGrwon_ZeaMays_RawPCD_100k.zip"
extract_path = "/content/raw_100k"

with zipfile.ZipFile(zip_path,'r') as zip_ref:
  zip_ref.extractall(extract_path)
print("Unzipped")

Unzipped


# Loading the point cloud files

In [ ]:
import os

data_path = "/content/raw_100k"

ply_files = []

for root, dirs, files in os.walk(data_path):
    for file in files:
        if file.endswith(".ply"):
            ply_files.append(os.path.join(root, file))

print("Total PLY files:", len(ply_files))
print("Example:", ply_files[0])

Total PLY files: 1045
Example: /content/raw_100k/FielGrwon_ZeaMays_RawPCD_100k/0870.ply


# Convert Point Cloud → Voxel Grid

We convert each plant into a 32×32×32 occupancy grid.

In [ ]:
VOXEL_RES = 32

def pointcloud_to_voxel(file):

    pcd = o3d.io.read_point_cloud(file)

    points = np.asarray(pcd.points)
    # normalize to unit cube
    points = points - points.min(0)
    points = points / points.max()

    grid = np.zeros((VOXEL_RES, VOXEL_RES, VOXEL_RES))

    coords = (points * (VOXEL_RES - 1)).astype(int)

    grid[coords[:,0], coords[:,1], coords[:,2]] = 1

    return grid

# Creating Dataset

In [ ]:
voxel_dataset = []

for file in tqdm(ply_files):
    vox = pointcloud_to_voxel(file)
    voxel_dataset.append(vox)

voxel_dataset = np.array(voxel_dataset)

print(voxel_dataset.shape)

100%|██████████| 1045/1045 [00:23<00:00, 45.19it/s]


(1045, 32, 32, 32)


# Converting to torch

In [ ]:
import torch

voxel_dataset = torch.tensor(voxel_dataset).float()
voxel_dataset = voxel_dataset.unsqueeze(1)

# Voxel Autoencoder Model

In [ ]:
import torch.nn as nn

class VoxelAutoencoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv3d(1,16,3,stride=2,padding=1),
            nn.ReLU(),

            nn.Conv3d(16,32,3,stride=2,padding=1),
            nn.ReLU(),

            nn.Conv3d(32,64,3,stride=2,padding=1),
            nn.ReLU()

        )

        self.decoder = nn.Sequential(

            nn.ConvTranspose3d(64,32,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose3d(32,16,3,stride=2,padding=1,output_padding=1),
            nn.ReLU(),

            nn.ConvTranspose3d(16,1,3,stride=2,padding=1,output_padding=1),
            nn.Sigmoid()

        )

    def forward(self,x):

        z = self.encoder(x)

        out = self.decoder(z)

        return out

# Training

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

loader = DataLoader(
    TensorDataset(voxel_dataset),
    batch_size=8,
    shuffle=True
)

model = VoxelAutoencoder().cuda()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

loss_fn = nn.BCELoss()

for epoch in range(40):

    total = 0

    for batch in loader:

        x = batch[0].cuda()

        optimizer.zero_grad()

        out = model(x)

        loss = loss_fn(out,x)

        loss.backward()

        optimizer.step()

        total += loss.item()

    print(epoch,total)

0 20.441706106066704
1 4.303348312154412
2 3.6456511225551367
3 3.250229135155678
4 3.0247099716216326
5 2.8879855889827013
6 2.786511603742838
7 2.7061755154281855
8 2.633574701845646
9 2.574242598377168
10 2.5079535953700542
11 2.449584028683603
12 2.387708542868495
13 2.323634095489979
14 2.2698755795136094
15 2.213425095193088
16 2.1647441200912
17 2.1106044771149755
18 2.0635139290243387
19 2.025629616342485
20 1.9816337125375867
21 1.9378360928967595
22 1.9052190696820617
23 1.8670925293117762
24 1.8316779481247067
25 1.7994163250550628
26 1.767827982082963
27 1.738064905628562
28 1.708074389025569
29 1.682097727432847
30 1.6533149322494864
31 1.631000827997923
32 1.6119993915781379
33 1.5899503780528903
34 1.5701103555038571
35 1.5475523481145501
36 1.5280621526762843
37 1.5096763269975781
38 1.4900453435257077
39 1.4773436384275556


In [ ]:
test = voxel_dataset[0].unsqueeze(0).cuda()

with torch.no_grad():
  reconstructed = model(test)

original = test.cpu().squeeze().numpy()
reconstructed = reconstructed.cpu().squeeze().numpy()

# Converting voxels to coordinates for visualizing

Voxel grids store occupancy values. We convert voxels > threshold into points.

In [ ]:
def voxel_to_points(voxel_grid, threshold=0.5):
    coords = np.argwhere(voxel_grid > threshold)
    return coords

In [ ]:
orig_pts = voxel_to_points(original)
recon_pts = voxel_to_points(reconstructed)

In [ ]:
recon_pts.shape

(424, 3)

#Plotting

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=orig_pts[:,0],
    y=orig_pts[:,1],
    z=orig_pts[:,2],
    mode='markers',
    marker=dict(size=3,color='blue'),
    name="Original Voxels"
))

fig.add_trace(go.Scatter3d(
    x=recon_pts[:,0],
    y=recon_pts[:,1],
    z=recon_pts[:,2],
    mode='markers',
    marker=dict(size=3,color='red'),
    name="Reconstructed Voxels"
))

fig.show()

# Saving and Loading model

In [ ]:
torch.save(model.state_dict(), "voxel_autoencoder.pth")

In [ ]:
# model = VoxelAutoencoder()   # define architecture first

# model.load_state_dict(
#     torch.load("voxel_autoencoder.pth")
# )

# model.eval()